In [1]:
print(123)

123


In [2]:
import pandas as pd
import numpy as np

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [4]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [7]:
print(assistant.rag("How do I run Ollama locally?"))

To run Ollama locally:

1. Install Ollama from **https://ollama.com/download** for your operating system.
   - **macOS**: download the `.pkg` and install it
   - **Windows**: download the `.msi` and install it
   - **Linux**: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

If you want to test that the local server is running, you can run:
```bash
curl http://localhost:11434
```

If you’re running into a connection refused error, restart the Ollama server with:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [10]:
assistant.rag("How do I run Olama locally?")

'The FAQ context only explains how to run the course locally, not Olama specifically.\n\nIf you mean running things locally in general, the answer is: you can run the course locally if you’re comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module. If you do, document your setup and keep it reproducible.\n\nIf you meant **Ollama**, I don’t have instructions for that in the provided context.'

In [11]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"},
    #{"role": "user", "content": "I just discovered Ollama. How do I run it locally?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Possibly — it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you figure it out quickly. Please send me:\n- the course name\n- where it’s hosted (school, platform, website)\n- whether you mean joining as a student now or asking for late enrollment\n\nIf you’re looking for a direct message to send, you could write:\n\n“Hi, I just discovered the course and I’m very interested in joining. Is it still possible to enroll, even though the course has already started?”\n\nIf you want, I can make that message more formal or more friendly.'

In [12]:
index.search("How do I run Ollama locally?")

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [13]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [14]:
search_results = search("I just discovered the course. Can I join it?")
print(search_results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {'id': '69d122f12e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?', 'answer': 'No, you can only get a certificate 

In [15]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

print(search_tool)

{'type': 'function', 'name': 'search', 'description': 'Search the FAQ database for entries matching the given query.', 'parameters': {'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the course FAQ.'}}, 'required': ['query'], 'additionalProperties': False}}


In [16]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join"}', call_id='call_CBPwkvGZP4YrYhEuFsaDICB1', name='search', type='function_call', id='fc_078f00279a89ac25006a37e09a1d1881a38f54dca8d5e6e1f4', namespace=None, status='completed')]

In [17]:
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join"}', call_id='call_CBPwkvGZP4YrYhEuFsaDICB1', name='search', type='function_call', id='fc_078f00279a89ac25006a37e09a1d1881a38f54dca8d5e6e1f4', namespace=None, status='completed')

In [18]:
import json 
args = json.loads(call.arguments)
args


{'query': 'join course discovered course can I join enrollment late join'}

In [19]:
results = search(**args)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [20]:
# Save the assistant's response with the tool call before overwriting response
assistant_response = response.output

response = search(**args)
result_json = json.dumps(response, indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "bd31146b0e",
    "course": "llm-zoomcamp",
    "section": "General Course

In [21]:
call.name

'search'

In [22]:
# Add the assistant's response with tool call to the conversation
messages.extend(assistant_response)

# Now add the function call output
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join"}', call_id='call_CBPwkvGZP4YrYhEuFsaDICB1', name='search', type='function_call', id='fc_078f00279a89ac25006a37e09a1d1881a38f54dca8d5e6e1f4', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_CBPwkvGZP4YrYhEuFsaDICB1',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificat

In [23]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

print(response.output_text)

Yes — you can still join the course.

If you want to receive a certificate, make sure you submit your project while submissions are still being accepted.


In [24]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(771, 34)

In [25]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(620 , 31)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001116


In [26]:
################
# Lesson 1.14 #
################

In [29]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. 

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [52]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. \n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join access FAQ"}', call_id='call_khBUAiS5trxN8KNSFg91nr3n', name='search', type='function_call', id='fc_0d8db92faab3ea08006a37e2126610819cb0b2ac780a66dfc2', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join access F

In [37]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

In [38]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)
response

Response(id='resp_0d8db92faab3ea08006a37e211d388819c81cf5292cd522d50', created_at=1782047249.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join access FAQ"}', call_id='call_khBUAiS5trxN8KNSFg91nr3n', name='search', type='function_call', id='fc_0d8db92faab3ea08006a37e2126610819cb0b2ac780a66dfc2', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the course FAQ.'}}, 'required': ['query'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Search the FAQ database for entries matching the given query.')], top_p=0.98, background=False, completed_at=1782047250

In [47]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [58]:
messages.extend(response.output)

for item in response.output:
    print(item)
    if item.type == "function_call":
        #process function call
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == "message":
        print("ASSISTANT:")
        print("Content:", item.content)

ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join access FAQ"}', call_id='call_khBUAiS5trxN8KNSFg91nr3n', name='search', type='function_call', id='fc_0d8db92faab3ea08006a37e2126610819cb0b2ac780a66dfc2', namespace=None, status='completed')
function_call: search {"query":"join course discovered course can I join enrollment late join access FAQ"}


In [61]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

iteration_count = 1
while True:
    print(f"Interaction {iteration_count}:")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool]
    )
    response

    messages.extend(response.output)

    for item in response.output:
        print(item)
        if item.type == "function_call":
            #process function call
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print("Content:", item.content)
    
    iteration_count += 1

    if has_function_calls == False  :
        break
    

Interaction 1:
ResponseFunctionToolCall(arguments='{"query":"join the course enroll discovered course can I join"}', call_id='call_zp4CHSRib6wDCQuBdeTtHyN9', name='search', type='function_call', id='fc_0824530024f702ba006a37e7864400819ca2c629d49ddad6f7', namespace=None, status='completed')
function_call: search {"query":"join the course enroll discovered course can I join"}
Interaction 2:
ResponseOutputMessage(id='msg_0824530024f702ba006a37e78762c8819c9897de27e09a6ca0', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, make sure to submit your project while submissions are still open. If you’re just learning, you can start anytime since the videos and materials are available.\n\nIf you want, I can also tell you how the course workflow works or what you need for the certificate.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')
ASSISTANT:
Content: [ResponseOu

In [69]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform the search , analyze the results, and then perform additional searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

def agent_loop(instructions, question, model="gpt-5.4-mini"): 
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    iteration_count = 1
    while True:
        print(f"Interaction {iteration_count}:")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )
        response

        messages.extend(response.output)

        for item in response.output:
            print(item)
            if item.type == "function_call":
                #process function call
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                print("Content:", item.content)
                last_assistant_message = item.content
        
        iteration_count += 1

        if has_function_calls == False  :
            break
    return last_assistant_message

response = agent_loop(instructions, question, model="gpt-5.4-mini")

print("Final response from the assistant:")
print(response)


Interaction 1:
ResponseFunctionToolCall(arguments='{"query":"join course enrollment discovered course can I join late enrollment registration new student"}', call_id='call_GXA454P91YE2qwMYWA9Lblcs', name='search', type='function_call', id='fc_0cb2eeca7f7bbf4f006a37e997a3c081a087775d5fe9f74558', namespace=None, status='completed')
function_call: search {"query":"join course enrollment discovered course can I join late enrollment registration new student"}
Interaction 2:
ResponseOutputMessage(id='msg_0cb2eeca7f7bbf4f006a37e998c07c81a094d485baf9b5bc4e', content=[ResponseOutputText(annotations=[], text='Yes — you can still join and start learning.\n\nIf you want a certificate, though, you’ll need to submit your project while the course is still accepting submissions. Otherwise, you can follow along in self-paced mode, but certificates aren’t available for that.\n\nIf you want, I can also help with how registration, homework, or the certificate process works.', type='output_text', logprobs=

In [70]:
question = "whats the course about?"
response = agent_loop(instructions, question, model="gpt-5.4-mini")

print("Final response from the assistant:")
print(response)


Interaction 1:
ResponseFunctionToolCall(arguments='{"query":"course about overview syllabus introduction"}', call_id='call_wTa04vV8nYPVG7uWjPD0IalL', name='search', type='function_call', id='fc_0c82c960853c1381006a37e9be57c081918fc1b7b8e051699b', namespace=None, status='completed')
function_call: search {"query":"course about overview syllabus introduction"}
Interaction 2:
ResponseFunctionToolCall(arguments='{"query":"LLM Zoomcamp what is the course about topics machine learning LLM agents RAG fine-tuning"}', call_id='call_2cpqRcZYtkROLPkps59zgkhL', name='search', type='function_call', id='fc_0c82c960853c1381006a37e9bf43088191b81f5096f6655060', namespace=None, status='completed')
function_call: search {"query":"LLM Zoomcamp what is the course about topics machine learning LLM agents RAG fine-tuning"}
Interaction 3:
ResponseFunctionToolCall(arguments='{"query":"LLM Zoomcamp syllabus topics modules RAG agentic RAG fine-tuning evaluation deployment"}', call_id='call_g3FMaUTea8zWeTbFkEA3XC

In [71]:
question = "how os the weather in london"
response = agent_loop(instructions, question, model="gpt-5.4-mini")

print("Final response from the assistant:")
print(response)

Interaction 1:
ResponseFunctionToolCall(arguments='{"query":"weather in London current forecast FAQ"}', call_id='call_GBYLckKvkqmTBNMs19Rn1zMA', name='search', type='function_call', id='fc_07465ee9cc9b891a006a37e9e49f8c819e98d250493df59d82', namespace=None, status='completed')
function_call: search {"query":"weather in London current forecast FAQ"}
Interaction 2:
ResponseFunctionToolCall(arguments='{"query":"London weather FAQ forecast climate temperature rain"}', call_id='call_T7opR7wpCLQ9vSGgdigL3n6R', name='search', type='function_call', id='fc_07465ee9cc9b891a006a37e9e56f84819ebb2d320876abc437', namespace=None, status='completed')
function_call: search {"query":"London weather FAQ forecast climate temperature rain"}
Interaction 3:
ResponseOutputMessage(id='msg_07465ee9cc9b891a006a37e9e64e10819e822cb7d2106cae8c', content=[ResponseOutputText(annotations=[], text='I couldn’t find a course FAQ entry that gives London’s actual weather right now.\n\nIf you want, I can help you with:\n- a

In [68]:
response = agent_loop(instructions, question, model="gpt-5.4-mini")

print("Final response from the assistant:")
print(response)

Interaction 1:
ResponseFunctionToolCall(arguments='{"query":"join the course enroll enrollment discover course can I join"}', call_id='call_54oyvf98u4lEqxhRqu0JTc1q', name='search', type='function_call', id='fc_04605fef690e3b38006a37e9583cec81a3b64f5724969da619', namespace=None, status='completed')
function_call: search {"query":"join the course enroll enrollment discover course can I join"}
Interaction 2:
ResponseFunctionToolCall(arguments='{"query":"certificate live cohort peer-review project submission still accepting submissions course running self-paced mode join course"}', call_id='call_FpfI3envqzZSdzbUumGdV7LO', name='search', type='function_call', id='fc_04605fef690e3b38006a37e959217881a3972765dbc84b5db9', namespace=None, status='completed')
function_call: search {"query":"certificate live cohort peer-review project submission still accepting submissions course running self-paced mode join course"}
Interaction 3:
ResponseOutputMessage(id='msg_04605fef690e3b38006a37e95a540881a3b

In [72]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

Interaction 1:
ResponseFunctionToolCall(arguments='{"query":"queen gambit"}', call_id='call_WmbbOqymE0masdoUaUwGCif6', name='search', type='function_call', id='fc_078e76c276236b21006a37ea8eb2ec81a1bd797dc9ec364e22', namespace=None, status='completed')
function_call: search {"query":"queen gambit"}
Interaction 2:
ResponseFunctionToolCall(arguments='{"query":"queen\'s gambit chess opening"}', call_id='call_rLgyddNyhgVzPMlEz9sqKMH8', name='search', type='function_call', id='fc_078e76c276236b21006a37ea8fafc081a1a2808d99d5edf5cf', namespace=None, status='completed')
function_call: search {"query":"queen's gambit chess opening"}
Interaction 3:
ResponseOutputMessage(id='msg_078e76c276236b21006a37ea906c1481a1b4993bba74379884', content=[ResponseOutputText(annotations=[], text='I couldn’t find a course FAQ entry for “queen gambit,” so it may be off-topic for this course. If you meant something course-related, feel free to rephrase it with more context.\n\nAre there other areas you want to explor

[ResponseOutputText(annotations=[], text='I couldn’t find a course FAQ entry for “queen gambit,” so it may be off-topic for this course. If you meant something course-related, feel free to rephrase it with more context.\n\nAre there other areas you want to explore?', type='output_text', logprobs=[])]